In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots

In [ ]:
from generate_data import gen_2d_data, load_image_point_cloud
from diffusion_geometry import DiffusionGeometry

Create some data

In [ ]:
data1 = load_image_point_cloud("../data/data11.jpeg", n=500, threshold=0.95, intensity_weighted=False)

plot_scatter_2d(data1, color = 'black').show()
dg1 = DiffusionGeometry.from_point_cloud(data1)

In [ ]:
# Up Hodge energy <da,db>
UH1 = dg1.up_laplacian(1)
DH1 = dg1.down_laplacian(1)

# Hodge energy <da,db> + <d^*a,d^*b>
H1 = UH1 + 1e+10 * DH1
vals_1, forms_1 = H1.spectrum()
print(vals_1[:10].round(2))

quiver_scale = 0.03
# arrow_scale = 0.3
line_width = 1.

a = forms_1[0]
f11 = plot_quiver_2d(data1, a.to_ambient(), scale=quiver_scale, line_width=line_width)
f11.show()

b = forms_1[1]
f12 = plot_quiver_2d(data1, b.to_ambient(), scale=quiver_scale, line_width=line_width)
f12.show()

c = forms_1[2]
f13 = plot_quiver_2d(data1, c.to_ambient(), scale=quiver_scale, line_width=line_width)
f13.show()

In [ ]:
def circular_coordinates(form, lam=1):
    dg = form.dg

    # Form the differential operator X - λΔ and compute its spectrum
    operator = form.sharp().operator - lam * dg.laplacian(0)
    evals, efunctions = operator.spectrum()

    # Select eigenfunctions with positive imaginary part (circular coordinates)
    circular_indices = np.where(evals.imag > 0)[0]
    circular_efunctions = efunctions[circular_indices].to_ambient()

    # Compute angles from the complex eigenfunctions
    angles = np.arctan2(circular_efunctions.imag, circular_efunctions.real)
    angles = np.mod(angles, 2 * np.pi)
    return angles

def opacity(form):
    op = form.pointwise_norm()**1
    op /= np.max(op)
    op = 2 * np.clip(op, 0, 0.5)
    return op

In [ ]:
colorscale = "hsv"
lam = 0.3
scatter_size = 4

circ_coords = circular_coordinates(a, lam = lam)
f11_c = plot_scatter_2d(data1, color="lightgray", size= scatter_size)
f11_c.add_traces(plot_scatter_2d(data1, color=circ_coords[0], opacity=opacity(a), colorscale=colorscale, cyclic=True, size = scatter_size).data)
f11_c.show()

circ_coords = circular_coordinates(b, lam = lam)
f12_c = plot_scatter_2d(data1, color="lightgray", size= scatter_size)
f12_c.add_traces(plot_scatter_2d(data1, color=circ_coords[0], opacity=opacity(b), colorscale=colorscale, cyclic=True, size = scatter_size).data)
f12_c.show()

circ_coords = circular_coordinates(c, lam = lam)
f13_c = plot_scatter_2d(data1, color="lightgray", size= scatter_size)
f13_c.add_traces(plot_scatter_2d(data1, color=circ_coords[0], opacity=opacity(c), colorscale=colorscale, cyclic=True, size = scatter_size).data)
f13_c.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Define which subplots are 2D or 3D ---
specs = [
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
]

fig = make_subplots(
    rows=2,
    cols=3,
    specs=specs,
    horizontal_spacing=0.02,
    vertical_spacing=0.06,
)

# --- Add all traces ---
fig.add_traces(list(f11.data), rows=[1]*len(f11.data), cols=[1]*len(f11.data))
fig.add_traces(list(f12.data), rows=[1]*len(f12.data), cols=[2]*len(f12.data))
fig.add_traces(list(f13.data), rows=[1]*len(f13.data), cols=[3]*len(f13.data))

fig.add_traces(list(f11_c.data), rows=[2]*len(f11_c.data), cols=[1]*len(f11_c.data))
fig.add_traces(list(f12_c.data), rows=[2]*len(f12_c.data), cols=[2]*len(f12_c.data))
fig.add_traces(list(f13_c.data), rows=[2]*len(f13_c.data), cols=[3]*len(f13_c.data))

clean_fig(fig)

# --- Synchronize 2D axis ranges (rows 1–2) ---
all_figs_2d = [f11, f12, f13, f11_c, f12_c, f13_c]
x_vals, y_vals = [], []
for f in all_figs_2d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, "y") and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])
xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]
fig.update_xaxes(range=xrange)
fig.update_yaxes(range=yrange)


# --- Layout cleanup ---
fig.update_layout(width=1100, height=700)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

fig.show()
fig.write_image("figs/new_output.pdf", scale=1)


In [ ]:
def labels(row, col):
    if row == 1:
        return rf"harmonic 1-form $\alpha_{col}$"
    elif row == 2:
        return rf"circular coordinate for $\alpha_{col}$"
    else:
        return ""

overpic_labels(fig, labels, 
               stretch_x=1, 
               stretch_y=1,
               offset_y=16.5)


In [ ]:
from generate_data import gen_3d_data

data2 = gen_3d_data(kind = 'torus', minor_radius = 1.0, major_radius=2.0, n = 1000, noise = 0.0, seed = 0)[0]
# data2 = gen_3d_data(kind = 'twoholed_torus', minor_radius = 1.0, major_radius=3.0, offset = 3.3, n = 5000, noise = 0.01, seed = 0)[0]
# data2 = data2[:, [1, 2, 0]]


camera2 = dict(eye=dict(x=2, y=0.4, z=1.6),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1))


fig = plot_scatter_3d(data2, color = 'black')
fig.update_layout(scene_camera=camera2)
fig.update_layout(width=400, height=450)
fig.show()

dg2 = DiffusionGeometry.from_point_cloud(data2)

In [ ]:
vals_1, forms_1 = dg2.laplacian(1).spectrum()
print(vals_1[:10].round(2))

quiver_scale = 0.12
# arrow_scale = 0.3
line_width = 1.5

a = forms_1[0]
f21 = plot_quiver_3d(data2, a.to_ambient(), scale=quiver_scale, arrow_scale=0.3, line_width=line_width)
f21.show()

b = forms_1[1]
f22 = plot_quiver_3d(data2, b.to_ambient(), scale=quiver_scale, arrow_scale=0.8, line_width=line_width)
f22.show()


colorscale = "hsv"
lam = 1
scatter_size = 2

circ_coords = circular_coordinates(a, lam = lam)
f21_c = plot_scatter_3d(data2, color=circ_coords[0], colorscale=colorscale, cyclic=True, camera=camera2, size = scatter_size)
f21_c.show()

circ_coords = circular_coordinates(b, lam = lam)
f22_c = plot_scatter_3d(data2, color=circ_coords[1], colorscale=colorscale, cyclic=True, camera=camera2, size = scatter_size)
f22_c.show()

In [ ]:
vals_2, forms_2 = dg2.laplacian(2).spectrum()
print(vals_2[:10].round(2))

quiver_scale = 0.15
# arrow_scale = 0.3
line_width = 1.5

harmonic = -forms_2[0]
f23 = plot_2form_3d(data2, harmonic.to_ambient())
f23.show()

wedge2 = a ^ b
fwedge2 = plot_2form_3d(data2, wedge2.to_ambient())
fwedge2.show()

print("Wedge norm:", wedge2.norm())
print("< wedge, harmonic 2-form >", dg2.inner(wedge2, harmonic))

In [ ]:
from generate_data import gen_3d_data

data3 = gen_3d_data(kind = 'sphere_with_handles', n = 600, noise = 0.02, seed = 1, grid_angles = True, points_per_component = 0.9)[0]
# data2 = gen_3d_data(kind = 'twoholed_torus', minor_radius = 1.0, major_radius=3.0, offset = 3.3, n = 5000, noise = 0.01, seed = 0)[0]
# data2 = data2[:, [1, 2, 0]]


camera3 = dict(eye=dict(x=1.5, y=1.3, z=0.9),
                center=dict(x=0.2, y=0, z=0),
                up=dict(x=0, y=0, z=1))


fig = plot_scatter_3d(data3, color = 'black')
fig.update_layout(scene_camera=camera3)
fig.update_layout(width=400, height=450)
fig.show()

dg3 = DiffusionGeometry.from_point_cloud(data3)

In [ ]:
vals_1, forms_1 = dg3.laplacian(1).spectrum()
print(vals_1[:10].round(2))

quiver_scale = 0.04
arrow_scale = 0.6
line_width = 1.5

a = forms_1[0]
f31 = plot_quiver_3d(data3, a.to_ambient(), scale=quiver_scale, arrow_scale=arrow_scale, line_width=line_width)
f31.update_layout(scene_camera=camera2)
f31.show()

b = forms_1[1]
f32 = plot_quiver_3d(data3, b.to_ambient(), scale=quiver_scale, arrow_scale=arrow_scale, line_width=line_width)
f32.update_layout(scene_camera=camera2)
f32.show()


colorscale = "hsv"
lam = 2
scatter_size = 4

circ_coords = circular_coordinates(a, lam = lam)
f31_c = plot_scatter_3d(data3, color="lightgray", size= scatter_size * 0.5)
f31_c.add_traces(plot_scatter_3d(data3, color=circ_coords[0], size=scatter_size * opacity(a), colorscale=colorscale, cyclic=True, camera=camera2).data)
f31_c.show()

circ_coords = circular_coordinates(b, lam = lam)
f32_c = plot_scatter_3d(data3, color="lightgray", size= scatter_size * 0.5)
f32_c.add_traces(plot_scatter_3d(data3, color=circ_coords[0], size=scatter_size * opacity(b), colorscale=colorscale, cyclic=True, camera=camera2).data)
f32_c.show()

In [ ]:
UH2 = dg3.up_laplacian(2)
DH2 = dg3.down_laplacian(2)
H2 = UH2 + 1e+1 * DH2

vals_2, forms_2 = H2.spectrum()
print(vals_2[:10].round(2))

harmonic = -forms_2[0]
f33 = plot_2form_3d(data3, 0.7*harmonic.to_ambient(), n_circle=20)
f33.show()

wedge3 = a ^ b
fwedge3 = plot_2form_3d(data3, wedge3.to_ambient(), n_circle=15)
fwedge3.show()
print("Wedge norm:", wedge3.norm())
print("< wedge, harmonic 2-form >", dg3.inner(wedge3, harmonic))

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Define which subplots are 2D or 3D ---
specs = [
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    rows=4,
    cols=3,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.0,
)

# --- Add all traces ---
fig.add_traces(list(f21.data), rows=[1]*len(f21.data), cols=[1]*len(f21.data))
fig.add_traces(list(f22.data), rows=[1]*len(f22.data), cols=[2]*len(f22.data))
fig.add_traces(list(fwedge2.data), rows=[1]*len(fwedge2.data), cols=[3]*len(fwedge2.data))

fig.add_traces(list(f21_c.data), rows=[2]*len(f21_c.data), cols=[1]*len(f21_c.data))
fig.add_traces(list(f22_c.data), rows=[2]*len(f22_c.data), cols=[2]*len(f22_c.data))
fig.add_traces(list(f23.data), rows=[2]*len(f23.data), cols=[3]*len(f23.data))

fig.add_traces(list(f31.data), rows=[3]*len(f31.data), cols=[1]*len(f31.data))
fig.add_traces(list(f32.data), rows=[3]*len(f32.data), cols=[2]*len(f32.data))
fig.add_traces(list(fwedge3.data), rows=[3]*len(fwedge3.data), cols=[3]*len(fwedge3.data))

fig.add_traces(list(f31_c.data), rows=[4]*len(f31_c.data), cols=[1]*len(f31_c.data))
fig.add_traces(list(f32_c.data), rows=[4]*len(f32_c.data), cols=[2]*len(f32_c.data))
fig.add_traces(list(f33.data), rows=[4]*len(f33.data), cols=[3]*len(f33.data))

# --- Compute separate ranges for torus (rows 3–4) and sphere+handles (rows 5–6) ---
def get_3d_bounds(figs):
    x, y, z = [], [], []
    for f in figs:
        for t in f.data:
            if hasattr(t, "x") and t.x is not None: x.extend(t.x)
            if hasattr(t, "y") and t.y is not None: y.extend(t.y)
            if hasattr(t, "z") and t.z is not None: z.extend(t.z)
    return [min(x), max(x)], [min(y), max(y)], [min(z), max(z)]

# Torus plots (rows 3–4)
torus_figs = [f21, f22, fwedge2, f21_c, f22_c, f23]
x_torus, y_torus, z_torus = get_3d_bounds(torus_figs)

# Sphere-with-handles plots (rows 5–6)
sphere_figs = [f31, f32, fwedge3, f31_c, f32_c, f33]
x_sphere, y_sphere, z_sphere = get_3d_bounds(sphere_figs)

# --- Apply ranges + cameras ---
for i in range(1, 13):
    key = f"scene{i}" if i > 1 else "scene"
    if i <= 6:  # rows 3–4 (torus)
        zoom_factor = 0.68
        camera_2_zoomed = dict(eye=dict(x=camera2['eye']['x']*zoom_factor,
                                         y=camera2['eye']['y']*zoom_factor,
                                         z=camera2['eye']['z']*zoom_factor),
                               center=camera2['center'],
                               up=camera2['up'])
        fig.update_layout({
            key: dict(
                camera=camera_2_zoomed,
                xaxis=dict(range=x_torus),
                yaxis=dict(range=y_torus),
                zaxis=dict(range=z_torus),
            )
        })
    else:        # rows 5–6 (sphere-with-handles)
        zoom_factor = 0.95
        camera3_zoomed = dict(eye=dict(x=camera3['eye']['x']*zoom_factor,
                                         y=camera3['eye']['y']*zoom_factor,
                                         z=camera3['eye']['z']*zoom_factor),
                               center=camera3['center'],
                               up=camera3['up'])
        fig.update_layout({
            key: dict(
                camera=camera3_zoomed,
                xaxis=dict(range=x_sphere),
                yaxis=dict(range=y_sphere),
                zaxis=dict(range=z_sphere),
            )
        })

# --- Layout cleanup ---
fig.update_layout(width=1000, height=1100)
clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

fig.show()
fig.write_image("figs/new_output.pdf", scale=1)


In [ ]:
def labels(row, col):
    if row == 1 or row == 3:
        if col == 1 or col == 2:
            return rf"harmonic 1-form $\alpha_{col}$, $\|\alpha_{col}\| = 1$"
        if col == 3:
            s = rf"$\alpha_1 \wedge \alpha_2$"
            if row == 1:
                s += rf", $\|\alpha_1 \wedge \alpha_2\| = 0.93$"
            if row == 3:
                s += rf", $\|\alpha_1 \wedge \alpha_2\| = 0.08$"
            return s
    elif row == 2 or row == 4:
        if col == 1 or col == 2:
            return rf"circular coordinate for $\alpha_{col}$"
        if col == 3:
            s = rf"harmonic 2-form $\beta$"
            if row == 2:
                s += rf", $\langle \alpha_1 \wedge \alpha_2, \beta \rangle = 0.54$"
            if row == 4:
                s += rf", $\langle \alpha_1 \wedge \alpha_2, \beta \rangle = 0.0002$"
            return s
    else:
        return ""

print('top two rows:')
overpic_labels(fig, labels, 
               stretch_x=1, 
               stretch_y=1,
               offset_y=14.5)

print('bottom two rows:')
overpic_labels(fig, labels, 
               stretch_x=1, 
               stretch_y=1,
               offset_y=10.5)

In [ ]:
data5 = np.random.uniform(-1, 1, (1400, 2)) * [1, 1]
plot_scatter_2d(data5, color = 'black').show()
dg5 = DiffusionGeometry.from_point_cloud(data5)

In [ ]:
x, y = data5.T
vortex1 = np.stack((-y-0.8, x-0.3), axis=1)
vortex1 /= np.linalg.norm(vortex1, axis=1, keepdims=True)**2 * 1

vortex2 = np.stack((y-0.8, -x-0.3), axis=1)
vortex2 /= np.linalg.norm(vortex2, axis=1, keepdims=True)**2 * 1

X = data5 - np.array([[-0.1, 0.7]])
X /= np.linalg.norm(X, axis=1, keepdims=True)**1.7 * 0.5

X = dg5.form(X, 1)

a = dg5.form(vortex1, 1)  - 1*X

exact, coexact, harmonic = a.hodge_decomposition()

scale = 0.02

f21 = plot_quiver_2d(data5, a.to_ambient(), scale=scale, line_width=1)
f21.show()

f22 = plot_quiver_2d(data5, exact.d().to_ambient(), scale=scale, line_width=1)
f22.show()

f23 = plot_quiver_2d(data5, coexact.codifferential().to_ambient(), scale=scale, line_width=1)
f23.show()

# f41 = plot_2form_2d(data5, a.d().to_ambient())
# f41.show()

# f42 = plot_quiver_2d(data5, a.up_laplacian().to_ambient(), scale=scale, line_width=line_width)
# f42.show()

# f43 = plot_quiver_2d(data5, a.laplacian().to_ambient(), scale=scale, line_width=line_width)
# f43.show()


In [ ]:
form = coexact.codifferential()

def circular_coordinates(form, lam=1):
    dg = form.dg

    # Form the differential operator X - λΔ and compute its spectrum
    operator = form.sharp().operator - lam * dg.laplacian(0)
    evals, efunctions = operator.spectrum()

    # Select eigenfunctions with positive imaginary part (circular coordinates)
    circular_indices = np.where(evals.imag > 0)[0]
    circular_efunctions = efunctions[circular_indices].to_ambient()
    circular_evals = evals[circular_indices]
    print("Selected eigenvalues (imag > 0):", circular_evals.round(3))
    
    # Compute angles from the complex eigenfunctions
    angles = np.arctan2(circular_efunctions.imag, circular_efunctions.real)
    angles = np.mod(angles, 2 * np.pi)
    return angles, circular_efunctions

lam = 0.5
angles, funcs = circular_coordinates(form, lam = lam)
func_mags = np.absolute(funcs)
k = 1
plot_quiver_2d(data5, form.to_ambient(), scale=0.02, line_width=1).show()
plot_scatter_2d(data5, color=func_mags[k], size=4).show()
plot_scatter_2d(data5, color=angles[k], colorscale="hsv", cyclic=True, size=4).show()

In [ ]:
np.absolute(funcs)

In [ ]:
circ_coords = circular_coordinates(form, lam = 1)
plot_scatter_2d(data5, color=circ_coords[0], colorscale="hsv", cyclic=True, size = 4).show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Define which subplots are 3D
specs = [
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    rows=2,
    cols=4,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.01,
)

# Add traces from each figure
fig.add_traces(list(f11.data), rows=[1]*len(f11.data), cols=[1]*len(f11.data))
fig.add_traces(list(f13.data), rows=[1]*len(f13.data), cols=[2]*len(f13.data))
fig.add_traces(list(f12.data), rows=[1]*len(f12.data), cols=[3]*len(f12.data))
fig.add_traces(list(f14.data), rows=[1]*len(f14.data), cols=[4]*len(f14.data))

fig.add_traces(list(f21.data), rows=[2]*len(f21.data), cols=[1]*len(f21.data))
fig.add_traces(list(f23.data), rows=[2]*len(f23.data), cols=[2]*len(f23.data))
fig.add_traces(list(f22.data), rows=[2]*len(f22.data), cols=[3]*len(f22.data))
fig.add_traces(list(f24.data), rows=[2]*len(f24.data), cols=[4]*len(f24.data))

# Synchronize axis ranges for top row (2D plots)
all_figs = [f11, f12, f13, f14]
x_vals = []
y_vals = []
for f in all_figs:
    for t in f.data:
        if hasattr(t, 'x') and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, 'y') and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])

xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]

# Apply to all top row subplots
fig.update_xaxes(range=xrange, row=1)
fig.update_yaxes(range=yrange, row=1)

# Synchronize axis ranges for bottom row (3D plots)
all_figs_3d = [f21, f22, f23, f24]
x_vals_3d = []
y_vals_3d = []
z_vals_3d = []
for f in all_figs_3d:
    for t in f.data:
        if hasattr(t, 'x') and t.x is not None:
            x_vals_3d.extend([v for v in t.x if v is not None])
        if hasattr(t, 'y') and t.y is not None:
            y_vals_3d.extend([v for v in t.y if v is not None])
        if hasattr(t, 'z') and t.z is not None:
            z_vals_3d.extend([v for v in t.z if v is not None])

xrange_3d = [min(x_vals_3d), max(x_vals_3d)]
yrange_3d = [min(y_vals_3d), max(y_vals_3d)]
zrange_3d = [min(z_vals_3d), max(z_vals_3d)]

# Apply to all bottom row 3D subplots
for i in range(1, 5):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        key: dict(
            camera=camera2,
            xaxis=dict(range=xrange_3d),
            yaxis=dict(range=yrange_3d),
            zaxis=dict(range=zrange_3d)
        )
    })

fig.update_layout(width=1000, height=800)
clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

fig.show()
fig.write_image("figs/new_output.pdf", scale=1)


In [ ]:

rows, cols = 2, 4
fig_w, fig_h = 1000, 800

def labels(row, col):
    if col == 1:
        return rf"$X$"
    elif col == 2:
        return "$\|X\| = \sqrt{g(X,X)}$"
    elif col == 3:
        return rf"$Y$"
    elif col == 4:
        return rf"$g(X,Y)$"
    else:
        return ""

overpic_labels(fig, labels, 
               stretch_x=0.98, 
               stretch_y=0.96,
               offset_y=-1.0)


In [ ]:
from figures.generate_data import load_image_point_cloud

data3 = load_image_point_cloud("../data/data1.jpeg", n=800, threshold=0.5, intensity_weighted=True)
data3 = data3[:, [1, 0]]

plot_scatter_2d(data3, color='black').show()

In [ ]:
dg3 = DiffusionGeometry.from_point_cloud(data3)

X = dg3.form_space(2).zeros()
X.coeffs[1] = 1
f31 = plot_2form_2d(data3, X.to_ambient())
f31.show()

Y = dg3.form_space(2).zeros()
Y.coeffs[4] = 1
f32 = plot_2form_2d(data3, Y.to_ambient())
f32.show()

norm_X = X.pointwise_norm()
f33 = plot_scatter_2d(data3, norm_X, size = 5)
f33.show()

gXY = dg3.g(X, Y)
f34 = plot_scatter_2d(data3, gXY, size = 5)
f34.show()


In [ ]:
data4 = gen_3d_data(kind = 'torus', minor_radius = 1.0, major_radius=2.0, n = 2000, noise = 0.0, seed = 0)[0]
data4 = data4[:, [1, 2, 0]]

k = 0.9
camera2_scaled = dict(eye=dict(x=camera2['eye']['x']*k, y=camera2['eye']['y']*k, z=camera2['eye']['z']*k),
                center=dict(x=camera2['center']['x'], y=camera2['center']['y']-0.05, z=camera2['center']['z']+0.07),
                up=camera2['up'])


fig = plot_scatter_3d(data4, color = 'black', camera=camera2_scaled)
fig.update_layout(scene_camera=camera2_scaled)
fig.update_layout(width=400, height=450)
fig.show()

dg4 = DiffusionGeometry.from_point_cloud(data4)

In [ ]:
# Up Hodge energy <da,db>
UH2 = dg4.up_laplacian(2)
DH2 = dg4.down_laplacian(2)

# Hodge energy <da,db> + <d^*a,d^*b>
H2 = 1e-2 * UH2 + DH2
vals_2, forms_2 = H2.spectrum()

X = dg4.form_space(2).zeros()
X.coeffs[0] = 1
f41 = plot_2form_3d(data4, X.to_ambient())
f41.update_layout(scene_camera=camera2_scaled)
f41.show()

Y = forms_2[0]
f42 = plot_2form_3d(data4, Y.to_ambient())
f42.update_layout(scene_camera=camera2_scaled)
f42.show()

norm_X = X.pointwise_norm()
f43 = plot_scatter_3d(data4, norm_X, size = 3)
f43.update_layout(scene_camera=camera2_scaled)
f43.show()

gXY = dg4.g(X, Y)
f44 = plot_scatter_3d(data4, gXY, size = 3)
f44.update_layout(scene_camera=camera2_scaled)
f44.show()


In [ ]:
data5, _ = gen_3d_data("cone", n=2000, noise=0.0, height=1.0, radius=0.4, solid=True)
dg5 = DiffusionGeometry.from_point_cloud(data5)

camera5 = dict(eye=dict(x=1.4, y=1.0, z=0.8),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1))

fig = plot_scatter_3d(data5, color='black')
fig.update_layout(scene_camera=camera5)
# fig.update_layout(width=400, height=450)
fig.show()

In [ ]:
X = dg5.form_space(3).zeros()
X.coeffs[1] = 1
f51 = plot_3form_3d(data5, X.to_ambient(), camera=camera5)
f51.update_layout(scene_camera=camera5)
f51.show()

Y = dg5.form_space(3).zeros()
Y.coeffs[2] = 1
f52 = plot_3form_3d(data5, Y.to_ambient(), camera=camera5)
f52.update_layout(scene_camera=camera5)
f52.show()

norm_X = X.pointwise_norm()
f53 = plot_scatter_3d(data5, norm_X, size = 3)
f53.update_layout(scene_camera=camera5)
f53.show()

gXY = dg5.g(X, Y)
f54 = plot_scatter_3d(data5, gXY, size = 3)
f54.update_layout(scene_camera=camera5)
f54.show()


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

figure_rows = [
    [f31, f33, f32, f34],
    [f41, f43, f42, f44],
    [f51, f53, f52, f54],
]


def _is_3d_figure(fig_obj):
    for trace in fig_obj.data:
        trace_type = getattr(trace, "type", "") or ""
        if "3d" in trace_type.lower():
            return True
        if hasattr(trace, "z") and trace.z is not None:
            return True
    return False


specs = []
scene_map = {}
scene_counter = 0
for row_idx, row_figs in enumerate(figure_rows, start=1):
    row_is_3d = any(_is_3d_figure(fig_obj) for fig_obj in row_figs)
    row_type = "scene" if row_is_3d else "xy"
    specs.append([{"type": row_type} for _ in row_figs])
    if row_type == "scene":
        for col_idx in range(1, len(row_figs) + 1):
            scene_counter += 1
            scene_name = "scene" if scene_counter == 1 else f"scene{scene_counter}"
            scene_map[(row_idx, col_idx)] = scene_name

fig = make_subplots(
    rows=len(figure_rows),
    cols=len(figure_rows[0]),
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.01,
)

for row_idx, row_figs in enumerate(figure_rows, start=1):
    for col_idx, src_fig in enumerate(row_figs, start=1):
        traces = list(src_fig.data)
        if not traces:
            continue
        fig.add_traces(
            traces,
            rows=[row_idx] * len(traces),
            cols=[col_idx] * len(traces),
        )

for row_idx, row_figs in enumerate(figure_rows, start=1):
    row_type = specs[row_idx - 1][0]["type"]
    if row_type == "xy":
        x_vals, y_vals = [], []
        for src_fig in row_figs:
            for trace in src_fig.data:
                if hasattr(trace, "x") and trace.x is not None:
                    x_vals.extend(v for v in trace.x if v is not None)
                if hasattr(trace, "y") and trace.y is not None:
                    y_vals.extend(v for v in trace.y if v is not None)
        if x_vals and y_vals:
            fig.update_xaxes(range=[min(x_vals), max(x_vals)], row=row_idx)
            fig.update_yaxes(range=[min(y_vals), max(y_vals)], row=row_idx)
    else:
        x_vals, y_vals, z_vals = [], [], []
        for src_fig in row_figs:
            for trace in src_fig.data:
                if hasattr(trace, "x") and trace.x is not None:
                    x_vals.extend(v for v in trace.x if v is not None)
                if hasattr(trace, "y") and trace.y is not None:
                    y_vals.extend(v for v in trace.y if v is not None)
                if hasattr(trace, "z") and trace.z is not None:
                    z_vals.extend(v for v in trace.z if v is not None)
        if x_vals and y_vals and z_vals:
            x_range = [min(x_vals), max(x_vals)]
            y_range = [min(y_vals), max(y_vals)]
            z_range = [min(z_vals), max(z_vals)]
            row_camera = None
            for src_fig in row_figs:
                scene_layout = getattr(src_fig.layout, "scene", None)
                if scene_layout and scene_layout.camera:
                    row_camera = scene_layout.camera
                    break
            if row_camera is None:
                row_camera = globals().get("camera")
            for col_idx in range(1, len(row_figs) + 1):
                scene_name = scene_map[(row_idx, col_idx)]
                scene_update = dict(
                    xaxis=dict(range=x_range),
                    yaxis=dict(range=y_range),
                    zaxis=dict(range=z_range),
                )
                if row_camera is not None:
                    scene_update["camera"] = row_camera
                fig.update_layout({scene_name: scene_update})

fig.update_layout(width=1000, height=1000)
clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

fig.show()
fig.write_image("figs/new_output_f345.pdf", scale=1)


In [ ]:
def labels(row, col):
    if col == 1:
        return rf"$\alpha$"
    elif col == 2:
        return rf"$\|\alpha\| = \sqrt{{g(\alpha,\alpha)}}$"
    elif col == 3:
        return rf"$\beta$"
    elif col == 4:
        return rf"$g(\alpha,\beta)$"
    else:
        return ""

overpic_labels(fig, labels, 
               stretch_x=0.98, 
               stretch_y=0.93,
               offset_y=16.5,
               offset_x=-0.)
